# 📈 04. Model Evaluation & Explainable AI (SHAP)
### TY BSc Data Science Final-Year Project
**Objective:** Evaluate test set performance across Accuracy, Precision, Recall, F1, and ROC-AUC, plot Confusion Matrices and ROC curves, and demonstrate SHAP explainability.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.append(os.path.abspath(".."))
from src.preprocessing import prepare_training_data
from src.evaluate_model import compute_evaluation_metrics
from src.explain import ChurnExplainer
from src.predict import ChurnPredictor, PRESET_PROFILES

sns.set_theme(style="whitegrid")
print("Imports ready.")

## 1. Load Models & Test Data

In [2]:
raw_data_path = os.path.join("..", "data", "raw", "customer_churn.csv")
X_train, X_test, y_test_labels, y_test, preprocessor, feature_names, _ = prepare_training_data(raw_data_path)

all_models = joblib.load(os.path.join("..", "models", "all_models.pkl"))
print(f"Loaded {len(all_models)} models.")

## 2. Test Set Evaluation Metrics Comparison

In [3]:
eval_results = []
for name, model in all_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_evaluation_metrics(y_test, y_pred, y_prob, model_name=name)
    eval_results.append(metrics)

eval_df = pd.DataFrame(eval_results)[['model_name', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]
eval_df

## 3. Best Model Confusion Matrix

In [4]:
best_model = joblib.load(os.path.join("..", "models", "churn_model.pkl"))
y_pred_best = best_model.predict(X_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Retained', 'Predicted Churn'],
            yticklabels=['Actual Retained', 'Actual Churn'])
plt.title("Best Model Confusion Matrix", fontsize=14, fontweight='bold')
plt.ylabel("Actual Status")
plt.xlabel("Predicted Status")
plt.show()

## 4. Explainable AI: SHAP Feature Attributions

In [5]:
explainer = ChurnExplainer()
predictor = ChurnPredictor()

test_profile = PRESET_PROFILES["High Risk (New Customer, Month-to-Month, High Cost)"]
pred_res = predictor.predict_single(test_profile)

print(f"Customer Churn Risk: {pred_res['risk_level'].upper()}")
print(f"Probability: {pred_res['probability_percent']}")
print("\nTop Drivers Increasing Churn Risk:")
for d in pred_res['risk_drivers'][:5]:
    print(f"  + {d['description']}: {d['shap_value']:+.4f}")